In [1]:
# ============================================================
# DAILY CLIMATE DATA COLLECTOR AGENT
# Fetch → Validate → Store → Log
# ============================================================

import os
import requests
import pandas as pd
import sqlite3
import logging
from datetime import datetime, timedelta


# ============================================================
# CONFIGURATION
# ============================================================

LAT = 16.7050   # example: Kolhapur latitude
LON = 74.2433   # Kolhapur longitude

DATABASE = "database/climate.db"
DATA_FOLDER = "data"
LOG_FOLDER = "logs"


# ============================================================
# CREATE REQUIRED FOLDERS
# ============================================================

os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(LOG_FOLDER, exist_ok=True)
os.makedirs("database", exist_ok=True)


# ============================================================
# LOGGING SETUP
# ============================================================

logging.basicConfig(
    filename=f"{LOG_FOLDER}/collector.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Climate Data Agent Started")


# ============================================================
# FETCH CLIMATE DATA
# ============================================================

def fetch_climate_data():

    logging.info("Fetching climate data from Open-Meteo")

    yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={LAT}&longitude={LON}"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum"
        f"&start_date={yesterday}&end_date={yesterday}"
        f"&timezone=auto"
    )

    response = requests.get(url)

    if response.status_code != 200:
        logging.error("API request failed")
        raise Exception("Failed API request")

    data = response.json()

    df = pd.DataFrame({
        "date": data["daily"]["time"],
        "temp_max": data["daily"]["temperature_2m_max"],
        "temp_min": data["daily"]["temperature_2m_min"],
        "precipitation": data["daily"]["precipitation_sum"]
    })

    logging.info("Climate data fetched successfully")

    return df


# ============================================================
# VALIDATE DATA
# ============================================================

def validate_data(df):

    logging.info("Validating data")

    if df.isnull().values.any():
        logging.warning("Missing values detected")

    df = df.drop_duplicates()

    return df


# ============================================================
# STORE DATA IN CSV
# ============================================================

def store_csv(df):

    csv_file = f"{DATA_FOLDER}/climate_data.csv"

    if os.path.exists(csv_file):
        existing = pd.read_csv(csv_file)
        df = pd.concat([existing, df]).drop_duplicates()
    
    df.to_csv(csv_file, index=False)

    logging.info("Data stored in CSV")


# ============================================================
# STORE DATA IN SQLITE
# ============================================================

def store_sqlite(df):

    conn = sqlite3.connect(DATABASE)

    df.to_sql(
        "daily_climate",
        conn,
        if_exists="append",
        index=False
    )

    conn.close()

    logging.info("Data stored in SQLite database")


# ============================================================
# MAIN AGENT
# ============================================================

def run_agent():

    try:

        df = fetch_climate_data()

        df = validate_data(df)

        store_csv(df)

        store_sqlite(df)

        print("Climate data stored successfully")

        logging.info("Agent completed successfully")

    except Exception as e:

        logging.error(f"Agent failed: {e}")

        print("Agent failed:", e)


# ============================================================
# RUN SCRIPT
# ============================================================

if __name__ == "__main__":

    run_agent()

Climate data stored successfully
